# 08 — RQ6: Is Austria's GHG Trajectory On Track for Its 2030 Emissions Target?

**Question:** Is Austria on track to meet its binding 2030 greenhouse-gas target — the EU
Effort Sharing Regulation's **−48% vs 2005 for non-ETS sectors** (Reg 2018/842, raised from
−36% by Reg 2023/857)?

Strictly the economy-wide **non-ETS emissions** question. The renewable-**electricity**
trajectory is RQ5, kept separate by design — one is an electricity-mix question, this is an
emissions question.

**Why a dedicated series.** The −48% target binds only the ~63% **non-ETS** slice (transport,
buildings, agriculture, small industry, waste), and that slice **cannot** be derived from
Eurostat's CRF sectors — the ETS/non-ETS boundary cuts across them. So the verdict uses the
EEA's Effort

In [ ]:
# Setup — imports, paths, house style, read-only DuckDB connection
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from matplotlib.lines import Line2D

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.viz import set_house_style, save_headline_fig

set_house_style()

# read_only=True — RQ notebooks only SELECT; a stray write raises instead of mutating the DB
DB_PATH = PROJECT_ROOT / "data" / "processed" / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)


In [ ]:
# Cell A: load ESR series + AEA path, define target
# RQ6: Is Austria's non-ETS (Effort Sharing) GHG trajectory on track for its 2030 target?
# Strictly the economy-wide non-ETS emissions question — kept distinct from RQ5 (renewable electricity).

AEA_CSV = PROJECT_ROOT / "data" / "reference" / "austria_esr_aea.csv"

# ── Constants (AR5 basis — matches the Effort Sharing Regulation target) ─────────
# Official ESR 2005 baseline = Annex I of Commission Implementing Decision 2020/2126.
# This is the AR5 figure (56.99 Mt); the EEA file's 2005 value (55.88 Mt) is the older
# ESD/AR4 baseyear — so every "% vs 2005" here uses 56.99 to stay on the target's basis.
BASELINE_2005_MT = 56.991984
TARGET_PCT       = 0.48                                 # -48% vs 2005 (Reg 2018/842, amended by 2023/857)
TARGET_2030_MT   = BASELINE_2005_MT * (1 - TARGET_PCT)  # ≈ 29.64 Mt

# ── Load: non-ETS emissions (history) and the binding AEA path ───────────────────
con = duckdb.connect(str(DB_PATH), read_only=True)      # read-only: notebook 08 only reads
esr = con.sql("""
    SELECT year, regime, mt_co2e
    FROM esr_emissions
    ORDER BY year
""").df()

aea = (pd.read_csv(AEA_CSV, comment="#")                # '#' provenance header skipped
         .rename(columns={"aea_mt_co2e": "aea_mt"}))

# Combined annual view 2005-2030: emissions where we have them, allocations 2021-2030
annual = (pd.merge(esr, aea[["year", "aea_mt"]], on="year", how="outer")
            .sort_values("year")
            .reset_index(drop=True))
annual["pct_vs_2005"] = 100 * (annual["mt_co2e"] / BASELINE_2005_MT - 1)

# ── Sanity checks ────────────────────────────────────────────────────────────────
# the 2030 AEA must equal the -48% target (consistency of the assembled path)
assert abs(aea.loc[aea.year == 2030, "aea_mt"].iloc[0] - TARGET_2030_MT) < 0.01

latest = esr.iloc[-1]
print(f"ESR emissions: {esr.year.min()}–{esr.year.max()}  ({len(esr)} yrs)")
print(f"AEA path:      {aea.year.min()}–{aea.year.max()}  ({len(aea)} yrs)")
print(f"2005 baseline (AR5): {BASELINE_2005_MT:.2f} Mt")
print(f"2030 target (-48%):  {TARGET_2030_MT:.2f} Mt  (= 2030 AEA)")
print(f"Latest ({int(latest.year)}): {latest.mt_co2e:.2f} Mt "
      f"({100*(latest.mt_co2e/BASELINE_2005_MT-1):+.1f}% vs 2005)")

annual

In [ ]:
# Cell B: log-linear (primary) + linear (robustness) trend fits
# across four windows. Log-linear => constant %/yr decline; linear => constant Mt/yr.

WINDOWS = [(2005, 2024), (2013, 2024), (2019, 2024), (2021, 2024)]
T0 = 2005                                    # shared time origin for all fits
T2030 = 2030 - T0

fits = {}                                    # window -> fitted models (kept for Cell C)
rows = []
for y0, y1 in WINDOWS:
    w = annual[(annual.year >= y0) & (annual.year <= y1) & annual.mt_co2e.notna()]
    t = (w.year - T0).to_numpy()
    X = sm.add_constant(t)
    e = w.mt_co2e.to_numpy()

    loglin = sm.OLS(np.log(e), X).fit()      # log-linear: emissions bounded below at 0
    lin    = sm.OLS(e, X).fit()              # linear: robustness comparison
    fits[f"{y0}–{y1}"] = {"loglin": loglin, "lin": lin, "w": w}

    loglin_2030 = float(np.exp(loglin.params[0] + loglin.params[1] * T2030))
    lin_2030    = float(lin.params[0] + lin.params[1] * T2030)
    rows.append({
        "window": f"{y0}–{y1}",
        "n": len(w),
        "decay_%/yr":          100 * (np.exp(loglin.params[1]) - 1),
        "loglin 2030 (Mt)":    loglin_2030,
        "linear 2030 (Mt)":    lin_2030,
        "% vs 2005":           100 * (loglin_2030 / BASELINE_2005_MT - 1),
        "gap vs target (Mt)":  loglin_2030 - TARGET_2030_MT,
    })

results = pd.DataFrame(rows).set_index("window").round(2)
print(f"Target 2030: {TARGET_2030_MT:.2f} Mt   (−48% vs {BASELINE_2005_MT:.2f} Mt, AR5 basis)\n")
results

In [ ]:
# Cell C: extrapolate the primary window with a prediction band,
# plus the full window fan. Log-linear fit => work in log space, exponentiate back to Mt.
PRIMARY = "2019–2024"
years_fwd = np.arange(2019, 2031)            # fit start → 2030
t_fwd = years_fwd - T0
Xf = sm.add_constant(t_fwd)

m = fits[PRIMARY]["loglin"]
pred = m.get_prediction(Xf).summary_frame(alpha=0.05)   # 95% intervals, in log space
proj = pd.DataFrame({
    "year":  years_fwd,
    "mean":  np.exp(pred["mean"]),                       # back to Mt
    "lower": np.exp(pred["obs_ci_lower"]),               # prediction interval (new obs)
    "upper": np.exp(pred["obs_ci_upper"]),
})

p2030 = proj[proj.year == 2030].iloc[0]
print(f"Primary window {PRIMARY}: 2030 central estimate {p2030['mean']:.1f} Mt "
      f"(95% prediction interval {p2030['lower']:.1f}–{p2030['upper']:.1f} Mt)")
print(f"Target {TARGET_2030_MT:.1f} Mt — clears target within band? "
      f"{'yes' if p2030['lower'] <= TARGET_2030_MT else 'no'}")

# all four windows projected to 2030 (central lines, for the fan)
fan = {}
for label, f in fits.items():
    fan[label] = np.exp(f["loglin"].params[0] + f["loglin"].params[1] * (years_fwd - T0))

proj.round(2)

In [ ]:
# Cell D: gap scorecard — required pace to hit 2030 vs recent pace
latest_year = int(esr.year.max())            # 2024
latest_mt   = float(esr.loc[esr.year == latest_year, "mt_co2e"].iloc[0])
n_left      = 2030 - latest_year             # 6 years

# Required constant-% cut from 2024 to land exactly on target in 2030
req_decay = (TARGET_2030_MT / latest_mt) ** (1 / n_left) - 1     # fractional/yr

# Recent achieved pace = the 2019–24 fitted decay (primary window)
recent_decay = np.exp(fits["2019–2024"]["loglin"].params[1]) - 1

print(f"From {latest_year} ({latest_mt:.1f} Mt) to 2030 target ({TARGET_2030_MT:.1f} Mt):")
print(f"  required pace : {100*req_decay:+.1f} %/yr")
print(f"  recent pace   : {100*recent_decay:+.1f} %/yr  (2019–24 fit)")
print(f"  → needs ~{req_decay/recent_decay:.1f}× the recent pace\n")

scorecard = (results[["loglin 2030 (Mt)", "% vs 2005", "gap vs target (Mt)"]]
             .rename(columns={"loglin 2030 (Mt)": "2030 (Mt)"}))
scorecard["% short of −48%"] = scorecard["% vs 2005"] + 48      # how far above the −48% line
scorecard.round(2)

In [ ]:
# Cell E: headline figure → figures/rq6_ghg_target_2030.png

C_EMIT, C_TREND, C_AEA = "#185FA5", "#D85A30", "#534AB7"     # blue / coral / purple
C_OVER, C_UNDER = "#C0392B", "#1D9E75"

fig, ax = plt.subplots(figsize=(10, 6))

# 1. non-primary windows — thin, recessive (the fan)
for label, line in fan.items():
    if label != PRIMARY:
        ax.plot(years_fwd, line, color="0.7", lw=0.9, ls=":", zorder=1)

# 2. forward gap (2024–30): extrapolation vs AEA budget = the miss
fwd = proj[proj.year >= 2024].merge(aea, on="year", how="left")
ax.fill_between(fwd.year, fwd["mean"], fwd["aea_mt"], color=C_TREND, alpha=0.07, zorder=1)

# 3. 95% prediction band + central extrapolation (primary 2019–24)
ax.fill_between(proj.year, proj.lower, proj.upper, color=C_TREND, alpha=0.12, zorder=2,
                label="95% prediction interval (2019–24)")
ax.plot(proj.year, proj["mean"], color=C_TREND, lw=2, ls="--", zorder=5,
        label="Trend extrapolation (2019–24)")

# 4. over / under allocation, 2021–24 — kept but unlabelled (too small to read at this scale)
b = annual[(annual.year >= 2021) & (annual.year <= 2024)][["year", "mt_co2e", "aea_mt"]].dropna()
ax.fill_between(b.year, b.mt_co2e, b.aea_mt, where=b.mt_co2e >= b.aea_mt, interpolate=True,
                color=C_OVER, alpha=0.35, zorder=3)
ax.fill_between(b.year, b.mt_co2e, b.aea_mt, where=b.mt_co2e <  b.aea_mt, interpolate=True,
                color=C_UNDER, alpha=0.35, zorder=3)

# 5. binding AEA path + 2030 target
ax.plot(aea.year, aea.aea_mt, color=C_AEA, lw=1.8, marker="s", ms=3, zorder=4,
        label="Binding allocation path (AEA)")
ax.scatter([2030], [TARGET_2030_MT], color=C_TREND, s=90, zorder=7, marker="D")
ax.annotate(f"−48% target  ({TARGET_2030_MT:.1f} Mt)", xy=(2030, TARGET_2030_MT),
            xytext=(2027, 33), ha="right", fontsize=9, color="#993C1D",
            arrowprops=dict(arrowstyle="->", color="#993C1D", lw=0.8))

# 6. historical emissions, split by accounting regime
esd, esr_ = esr[esr.regime == "ESD"], esr[esr.regime == "ESR"]
ax.plot(esd.year, esd.mt_co2e, color=C_EMIT, lw=2, marker="o", ms=4, zorder=6,
        label="Non-ETS emissions (ESD / AR4)")
ax.plot(esr_.year, esr_.mt_co2e, color=C_EMIT, lw=2, marker="o", ms=4, ls=(0, (1, 1)),
        zorder=6, label="Non-ETS emissions (ESR / AR5)")

ax.axvspan(2020.5, 2021.5, color="0.9", zorder=0)
ax.text(2021, 55.3, "AR4→AR5", fontsize=7, ha="center", color="0.5")

ax.set_xlabel("Year"); ax.set_ylabel("Non-ETS GHG emissions (Mt CO₂-eq)")
ax.set_title("Austria's non-ETS emissions vs the 2030 Effort Sharing target")
ax.set_ylim(28, 56); ax.set_xlim(2005, 2031)

handles, labels = ax.get_legend_handles_labels()
handles.append(Line2D([0], [0], color="0.7", lw=0.9, ls=":"))
labels.append("Other trend windows (2005–24, 2013–24, 2021–24)")
ax.legend(handles, labels, loc="lower left", fontsize=8, ncol=1, framealpha=0.9)

fig.tight_layout()
save_headline_fig(fig, "rq6_ghg_target_2030")
plt.show()

In [ ]:
# Close the connection
con.close()

Finding. On its post-2019 trend, Austria's non-ETS emissions are projected to reach ≈36.8 Mt by 2030 (95% prediction interval 29.8–45.4 Mt) — roughly 7 Mt above the binding −48% Effort Sharing target of 29.6 Mt, and short of it across all four trend windows tested (gap +2.6 to +13.9 Mt). Closing it would require cutting emissions at about −6%/yr, ~2.2× the recent pace, and the only window that nearly reaches the target leans entirely on energy-crisis-driven reductions whose permanence is unproven.